In [1]:
import os

In [2]:
from unstructured.partition.pdf import partition_pdf
from unstructured.partition.docx import partition_docx
from unstructured.documents.elements import Table, Image
from pathlib import Path

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Chuyển về folder gốc project
os.chdir(Path.cwd().parent)
print(os.getcwd())  # Phải in ra folder chứa Data/, Script/, .env

c:\Users\LENOVO\Documents\GitHub\Chatbot QCDT V2


In [4]:
import json
import logging
import traceback
from datetime import datetime
from typing import Dict, List, Tuple

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

In [17]:
def create_output_path(input_file_path: Path, base_output_dir: Path = Path("Partitioned_Data")) -> Path:
    """
    Create output path mirroring input structure.
    
    Example:
        Data/QCDT/QCDT_2025.pdf → Partitioned_Data/QCDT/QCDT_2025/
    
    Args:
        input_file_path: Full path to input file
        base_output_dir: Base output directory (default: Partitioned_Data)
    
    Returns:
        Path to output folder for this file
    """
    # Convert to absolute path to ensure consistency
    input_file_path = Path(input_file_path).resolve()
    base_output_dir = Path(base_output_dir)
    
    # Get relative path from Data/ folder
    data_path = Path("Data").resolve()
    
    try:
        relative_path = input_file_path.relative_to(data_path)
    except ValueError:
        # If not under Data/, try to extract from path parts
        # Find the position of "Data" in the path and extract everything after it
        parts = input_file_path.parts
        try:
            data_idx = parts.index("Data")
            relative_path = Path(*parts[data_idx + 1:])
        except (ValueError, IndexError):
            raise ValueError(f"Cannot determine relative path for {input_file_path}")
    
    # Remove file name and extension, use as folder name
    file_name_without_ext = relative_path.stem
    parent_dirs = relative_path.parent
    
    # Build output path: Partitioned_Data/{parent_dirs}/{filename}/
    output_path = base_output_dir / parent_dirs / file_name_without_ext
    
    return output_path

In [ ]:
def batch_partition(
    input_dir: Path = Path("Data"),
    base_output_dir: Path = Path("Partitioned_Data"),
    skip_existing: bool = True
) -> Dict[str, int]:
    """
    Batch partition all PDF and DOCX files in input directory recursively.
    
    Args:
        input_dir: Root input directory (default: Data)
        base_output_dir: Base output directory (default: Partitioned_Data)
        skip_existing: If True, skip already partitioned files
    
    Returns:
        Dict with counts: {total_processed, total_skipped, total_failed}
    """
    input_dir = Path(input_dir)
    
    if not input_dir.exists():
        logger.error(f"Input directory does not exist: {input_dir}")
        return {"total_processed": 0, "total_skipped": 0, "total_failed": 0}
    
    # Find all PDF and DOCX files
    pdf_files = list(input_dir.rglob("*.pdf"))
    docx_files = list(input_dir.rglob("*.docx"))
    all_files = sorted(pdf_files + docx_files)
    
    if not all_files:
        logger.warning(f"No PDF or DOCX files found in {input_dir}")
        return {"total_processed": 0, "total_skipped": 0, "total_failed": 0}
    
    logger.info(f"Found {len(all_files)} files to process")
    logger.info("=" * 80)
    
    stats = {"total_processed": 0, "total_skipped": 0, "total_failed": 0}
    
    # Process files sequentially
    for idx, file_path in enumerate(all_files, 1):
        file_path = file_path.resolve()  # Convert to absolute path
        logger.info(f"[{idx}/{len(all_files)}] Processing: {file_path.relative_to(input_dir)}")
        
        output_path = create_output_path(file_path, base_output_dir)
        
        # Check skip condition before processing
        if skip_existing and (output_path / "texts.json").exists():
            stats["total_skipped"] += 1
            logger.info(f"⊘ Skipped (already partitioned)")
        else:
            # Partition file
            success = partition_single_file(file_path, base_output_dir, skip_existing=False)
            if success:
                stats["total_processed"] += 1
            else:
                stats["total_failed"] += 1
        
        logger.info("-" * 80)
    
    # Print summary
    logger.info("=" * 80)
    logger.info("BATCH PROCESSING SUMMARY")
    logger.info("=" * 80)
    logger.info(f"Total files processed: {stats['total_processed']}")
    logger.info(f"Total files skipped:   {stats['total_skipped']}")
    logger.info(f"Total files failed:    {stats['total_failed']}")
    logger.info(f"Total files found:     {len(all_files)}")
    logger.info("=" * 80)
    
    return stats

## Example 1: Partition a Single File

Modify `test_file_path` to point to any PDF or DOCX file in the `Data/` folder.

In [10]:
# Example: Partition a single file
# Modify this path to any PDF or DOCX file in Data/ folder
test_file_path = Path(r"C:\Users\LENOVO\Documents\GitHub\Chatbot QCDT V2\Data\Ngoai_ngu\Ngoai_ngu_2022_Quy_doi_CCTA.pdf")

# Uncomment to test
if test_file_path.exists():
    success = partition_single_file(test_file_path)
    print(f"Partition result: {'Success' if success else 'Failed'}")
else:
    print(f"File not found: {test_file_path}")
    print(f"Available files in Data/:")
    for f in Path("Data").rglob("*.pdf"):
        print(f"  - {f}")
    for f in Path("Data").rglob("*.docx"):
        print(f"  - {f}")

2026-05-10 10:15:12 - INFO - Already partitioned: Ngoai_ngu_2022_Quy_doi_CCTA.pdf


Partition result: Success


In [12]:
import shutil

# Delete old output to reprocess
test_pdf = Path("Data/Ngoai_ngu/Ngoai_ngu_2022_Quy_doi_CCTA.pdf").resolve()
output_path = create_output_path(test_pdf)
if output_path.exists():
    shutil.rmtree(output_path)
    print(f"Deleted old output: {output_path}")

# Now test partition_single_file
if test_pdf.exists():
    success = partition_single_file(test_pdf)
    print(f"Partition result: {'✓ Success' if success else '✗ Failed'}")
else:
    print(f"File not found: {test_pdf}")

2026-05-10 10:15:37 - INFO - Starting partition: Ngoai_ngu_2022_Quy_doi_CCTA.pdf


Deleted old output: Partitioned_Data\Ngoai_ngu\Ngoai_ngu_2022_Quy_doi_CCTA


2026-05-10 10:15:38 - INFO - Downloading spaCy model en_core_web_sm 3.8.0 …
2026-05-10 10:15:41 - INFO - PDF text extraction failed, skip text extraction...
2026-05-10 10:15:41 - INFO - Reading PDF for file: C:\Users\LENOVO\Documents\GitHub\Chatbot QCDT V2\Data\Ngoai_ngu\Ngoai_ngu_2022_Quy_doi_CCTA.pdf ...
2026-05-10 10:16:16 - INFO - ✓ Partitioned Ngoai_ngu_2022_Quy_doi_CCTA.pdf: 23 texts, 2 tables, 0 images


Partition result: ✓ Success


## DEBUG: Test PDF Text Extraction

Debug cell để kiểm tra chi tiết lỗi khi extract text từ PDF file `Ngoai_ngu_2022_Quy_doi_CCTA.pdf`

In [9]:
# Test file path
test_pdf = Path("Data/Ngoai_ngu/Ngoai_ngu_2022_Quy_doi_CCTA.pdf")

print(f"Testing PDF file: {test_pdf}")
print(f"File exists: {test_pdf.exists()}")
print(f"File size: {test_pdf.stat().st_size / 1024:.2f} KB")
print("=" * 80)

# Step 1: Try basic partition_pdf without any parameters
print("\n[STEP 1] Basic partition_pdf() - no parameters")
try:
    elements = partition_pdf(filename=str(test_pdf))
    print(f"✓ Success! Got {len(elements)} elements")
    for i, el in enumerate(elements[:3]):
        print(f"  [{i}] {type(el).__name__}: {str(el.text)[:80]}")
except Exception as e:
    print(f"✗ Failed: {str(e)}")
    print(f"Error type: {type(e).__name__}")

# Step 2: Try with strategy="hi_res"
print("\n[STEP 2] partition_pdf with strategy='hi_res'")
try:
    elements = partition_pdf(
        filename=str(test_pdf),
        strategy="hi_res"
    )
    print(f"✓ Success! Got {len(elements)} elements")
    for i, el in enumerate(elements[:3]):
        print(f"  [{i}] {type(el).__name__}: {str(el.text)[:80]}")
except Exception as e:
    print(f"✗ Failed: {str(e)}")
    print(f"Error type: {type(e).__name__}")

# Step 3: Try with extract_image_block_types
print("\n[STEP 3] partition_pdf with extract_image_block_types=['Image', 'Table']")
try:
    elements = partition_pdf(
        filename=str(test_pdf),
        strategy="hi_res",
        extract_image_block_types=["Image", "Table"]
    )
    print(f"✓ Success! Got {len(elements)} elements")
    for i, el in enumerate(elements[:3]):
        print(f"  [{i}] {type(el).__name__}: {str(el.text)[:80]}")
except Exception as e:
    print(f"✗ Failed: {str(e)}")
    print(f"Error type: {type(e).__name__}")

# Step 4: Try with infer_table_structure
print("\n[STEP 4] partition_pdf with infer_table_structure=True")
try:
    elements = partition_pdf(
        filename=str(test_pdf),
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image", "Table"]
    )
    print(f"✓ Success! Got {len(elements)} elements")
    for i, el in enumerate(elements[:3]):
        print(f"  [{i}] {type(el).__name__}: {str(el.text)[:80]}")
except Exception as e:
    print(f"✗ Failed: {str(e)}")
    print(f"Error type: {type(e).__name__}")

# Step 5: Try with extract_image_block_to_payload
print("\n[STEP 5] partition_pdf with extract_image_block_to_payload=True")
try:
    elements = partition_pdf(
        filename=str(test_pdf),
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image", "Table"],
        extract_image_block_to_payload=True
    )
    print(f"✓ Success! Got {len(elements)} elements")
    for i, el in enumerate(elements[:3]):
        print(f"  [{i}] {type(el).__name__}: {str(el.text)[:80]}")
except Exception as e:
    print(f"✗ Failed: {str(e)}")
    print(f"Error type: {type(e).__name__}")

# Step 6: Try with languages parameter
print("\n[STEP 6] partition_pdf with languages=['vie']")
try:
    elements = partition_pdf(
        filename=str(test_pdf),
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image", "Table"],
        extract_image_block_to_payload=True,
        languages=["vie"]
    )
    print(f"✓ Success! Got {len(elements)} elements")
    for i, el in enumerate(elements[:3]):
        print(f"  [{i}] {type(el).__name__}: {str(el.text)[:80]}")
except Exception as e:
    print(f"✗ Failed: {str(e)}")
    print(f"Error type: {type(e).__name__}")

2026-05-10 10:12:31 - INFO - Downloading spaCy model en_core_web_sm 3.8.0 …


Testing PDF file: Data\Ngoai_ngu\Ngoai_ngu_2022_Quy_doi_CCTA.pdf
File exists: True
File size: 777.69 KB

[STEP 1] Basic partition_pdf() - no parameters


2026-05-10 10:12:33 - INFO - PDF text extraction failed, skip text extraction...
2026-05-10 10:12:33 - WARNING - No languages specified, defaulting to English.
2026-05-10 10:12:43 - INFO - Downloading spaCy model en_core_web_sm 3.8.0 …
2026-05-10 10:12:46 - INFO - Downloading spaCy model en_core_web_sm 3.8.0 …


✗ Failed: Failed to install en_core_web_sm to c:\python313\Lib\site-packages: [WinError 5] Access is denied: 'c:\\python313\\Lib\\site-packages\\en_core_web_sm'. Ensure the site-packages directory is writable, or pre-install the model with: python -m spacy download en_core_web_sm
Error type: RuntimeError

[STEP 2] partition_pdf with strategy='hi_res'


2026-05-10 10:12:48 - INFO - PDF text extraction failed, skip text extraction...
2026-05-10 10:12:48 - WARNING - No languages specified, defaulting to English.
2026-05-10 10:12:48 - INFO - Reading PDF for file: Data\Ngoai_ngu\Ngoai_ngu_2022_Quy_doi_CCTA.pdf ...


...

✓ Success! Got 25 elements
  [0] Header: BO GIAO DUC VA DAO TAO TRUONG DAI HOC BACH KHOA HA
  [1] Text: NOI
  [2] Title: CONG HOA XA HOI CHU NGHIA VIET NAM

[STEP 5] partition_pdf with extract_image_block_to_payload=True


2026-05-10 10:14:06 - INFO - PDF text extraction failed, skip text extraction...
2026-05-10 10:14:06 - WARNING - No languages specified, defaulting to English.
2026-05-10 10:14:06 - INFO - Reading PDF for file: Data\Ngoai_ngu\Ngoai_ngu_2022_Quy_doi_CCTA.pdf ...
2026-05-10 10:14:17 - INFO - Downloading spaCy model en_core_web_sm 3.8.0 …


✓ Success! Got 25 elements
  [0] Header: BO GIAO DUC VA DAO TAO TRUONG DAI HOC BACH KHOA HA
  [1] Text: NOI
  [2] Title: CONG HOA XA HOI CHU NGHIA VIET NAM

[STEP 6] partition_pdf with languages=['vie']


2026-05-10 10:14:19 - INFO - PDF text extraction failed, skip text extraction...
2026-05-10 10:14:19 - INFO - Reading PDF for file: Data\Ngoai_ngu\Ngoai_ngu_2022_Quy_doi_CCTA.pdf ...


✓ Success! Got 25 elements
  [0] Header: BỘ GIÁO DỤC VÀ ĐÀO TẠO TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ
  [1] Text: NỘI
  [2] Title: CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM


## Example 2: Batch Partition All Files

This will recursively find all PDF and DOCX files in `Data/` and partition them sequentially.

## TEST: Partition a Single DOCX File

Test file `Do_an/DATN_2021_Dang_ky.docx` trước khi batch partition tất cả file word

In [23]:
# Test DOCX file partition
test_docx = Path("Data/Do_an/DATN_2021_Dang_ky.docx").resolve()

print(f"Testing DOCX file: {test_docx.name}")
print(f"File exists: {test_docx.exists()}")

if test_docx.exists():
    print(f"File size: {test_docx.stat().st_size / 1024:.2f} KB")
    print("=" * 80)
    
    # Delete old output if exists
    output_path = create_output_path(test_docx)
    if output_path.exists():
        import shutil
        shutil.rmtree(output_path)
        print(f"Deleted old output: {output_path}")
    
    # Partition the file
    success = partition_single_file(test_docx)
    print(f"\nPartition result: {'✓ Success' if success else '✗ Failed'}")
    
    # Show output info if successful
    if success:
        print(f"\nOutput folder: {output_path}")
        for json_file in sorted(output_path.glob("*.json")):
            if json_file.name != "partition_done.json":
                with open(json_file, 'r', encoding='utf-8') as f:
                    content = json.load(f)
                    count = len(content) if isinstance(content, list) else 1
                file_size = json_file.stat().st_size / 1024
                print(f"  - {json_file.name}: {count} items ({file_size:.1f} KB)")
else:
    print(f"✗ File not found: {test_docx}")
    print("\nAvailable DOCX files in Data/:")
    for f in sorted(Path("Data").rglob("*.docx")):
        print(f"  - {f.relative_to('Data')}")

2026-05-10 12:19:42 - INFO - Starting partition: DATN_2021_Dang_ky.docx


2026-05-10 12:19:42 - INFO - ✓ Partitioned DATN_2021_Dang_ky.docx: 101 texts, 0 tables, 0 images


Testing DOCX file: DATN_2021_Dang_ky.docx
File exists: True
File size: 16.59 KB

Partition result: ✓ Success

Output folder: Partitioned_Data\Do_an\DATN_2021_Dang_ky
  - texts.json: 101 items (14.1 KB)


In [22]:
# DEBUG: Try direct partition_docx call to see error
print("DEBUG: Direct partition_docx call")
try:
    test_docx = Path("Data/Do_an/DATN_2021_Dang_ky.docx").resolve()
    elements = partition_docx(filename=str(test_docx))
    print(f"✓ Success! Got {len(elements)} elements")
    for i, el in enumerate(elements[:3]):
        print(f"  [{i}] {type(el).__name__}: {str(el.text)[:80]}")
except Exception as e:
    print(f"✗ Error: {str(e)}")
    print(f"Error type: {type(e).__name__}")
    import traceback
    print("\nFull traceback:")
    print(traceback.format_exc())

DEBUG: Direct partition_docx call
✓ Success! Got 101 elements
  [0] NarrativeText: Một số lưu ý khi sinh viên đăng ký ĐATN
  [1] Text: Sinh viên năm thứ 4 có các phương án lựa chọn tiếp tục học tập hoặc tốt nghiệp.
  [2] Text: 1. Mong muốn Tốt nghiệp Cử nhân


In [18]:
# Run batch partitioning
# Set skip_existing=True to skip files that already have texts.json
# Set skip_existing=False to reprocess all files
stats = batch_partition(
    input_dir=Path("Data"),
    base_output_dir=Path("Partitioned_Data"),
    skip_existing=True
)

2026-05-10 10:17:18 - INFO - Found 19 files to process
2026-05-10 10:17:18 - INFO - ================================================================================
2026-05-10 10:17:18 - INFO - [1/19] Processing: Do_an\DATN_2021_Dang_ky.docx
2026-05-10 10:17:18 - INFO - Starting partition: DATN_2021_Dang_ky.docx
2026-05-10 10:17:19 - INFO - Downloading spaCy model en_core_web_sm 3.8.0 …
2026-05-10 10:17:20 - ERROR - ✗ Failed to partition DATN_2021_Dang_ky.docx
2026-05-10 10:17:20 - ERROR - Error: Failed to install en_core_web_sm to c:\python313\Lib\site-packages: [WinError 5] Access is denied: 'c:\\python313\\Lib\\site-packages\\en_core_web_sm'. Ensure the site-packages directory is writable, or pre-install the model with: python -m spacy download en_core_web_sm
2026-05-10 10:17:20 - ERROR - Traceback:
Traceback (most recent call last):
  File "c:\python313\Lib\shutil.py", line 856, in move
    os.rename(src, real_dst)
    ~~~~~~~~~^^^^^^^^^^^^^^^
PermissionError: [WinError 5] Access i

## Inspect Output Structure

View the output folder structure and sample data from partitioned files.

In [19]:
from pathlib import Path
import json

# List all partitioned output folders
output_dir = Path("Partitioned_Data")
if output_dir.exists():
    print("Partitioned output folders:")
    print("=" * 80)
    
    for folder in sorted(output_dir.rglob("*")):
        if folder.is_dir():
            # Check what files are in this folder
            files_in_folder = list(folder.glob("*.json"))
            if files_in_folder:
                rel_path = folder.relative_to(output_dir)
                print(f"\n📁 {rel_path}")
                
                for json_file in sorted(files_in_folder):
                    file_size = json_file.stat().st_size / 1024  # KB
                    with open(json_file, 'r', encoding='utf-8') as f:
                        content = json.load(f)
                        count = len(content) if isinstance(content, list) else 1
                    print(f"   - {json_file.name}: {count} items ({file_size:.1f} KB)")
    
    print("\n" + "=" * 80)
    print(f"✓ Output directory: {output_dir.resolve()}")
else:
    print("No output directory found. Run batch partitioning first.")

Partitioned output folders:

📁 Hoc_bong\Hoc_bong_2021_TDN
   - images.json: 1 items (77.3 KB)
   - partition_done.json: 1 items (0.0 KB)
   - texts.json: 88 items (14.7 KB)

📁 Hoc_bong\Hoc_bong_2022_KKHT
   - images.json: 1 items (92.6 KB)
   - partition_done.json: 1 items (0.0 KB)
   - texts.json: 36 items (7.3 KB)

📁 Hoc_phi\Hoc_phi_2025_DHCQ_KSCS_VLVH_SDH
   - images.json: 1 items (21.9 KB)
   - partition_done.json: 1 items (0.0 KB)
   - tables.json: 8 items (2126.9 KB)
   - texts.json: 85 items (15.5 KB)

📁 Huong_dan\Huong_dan_2021_Quy_doi_tin_chi
   - images.json: 2 items (165.3 KB)
   - partition_done.json: 1 items (0.0 KB)
   - tables.json: 5 items (612.1 KB)
   - texts.json: 88 items (18.7 KB)

📁 Huong_dan\Huong_dan_2023_Quy_doi_tin_chi
   - images.json: 1 items (633.9 KB)
   - partition_done.json: 1 items (0.0 KB)
   - tables.json: 3 items (2501.3 KB)
   - texts.json: 24 items (1.9 KB)

📁 Huong_dan\Huong_dan_2023_Thu_tuc_hanh_chinh_VB_CC_KQHT
   - images.json: 1 items (20.4 KB

## Load and View Sample Data

Example of loading partitioned data from JSON files.

In [14]:
# Example: Load data from a partitioned file
# Modify this path to any partitioned folder

example_folder = Path("Partitioned_Data") / "Ngoai_ngu" / "Ngoai_ngu_2022_Quy_doi_CCTA"  # Change this path

if example_folder.exists():
    print(f"📂 Loading data from: {example_folder}")
    print("=" * 80)
    
    # Load texts
    texts_file = example_folder / "texts.json"
    if texts_file.exists():
        with open(texts_file, 'r', encoding='utf-8') as f:
            texts = json.load(f)
        print(f"\n📄 Texts ({len(texts)} items):")
        print("-" * 80)
        for i, item in enumerate(texts[:2]):  # Show first 2
            print(f"[{i}] Type: {item['element_type']}")
            print(f"    Text: {item['text'][:100]}..." if len(item['text']) > 100 else f"    Text: {item['text']}")
        if len(texts) > 2:
            print(f"    ... and {len(texts) - 2} more items")
    
    # Load tables
    tables_file = example_folder / "tables.json"
    if tables_file.exists():
        with open(tables_file, 'r', encoding='utf-8') as f:
            tables = json.load(f)
        print(f"\n📊 Tables ({len(tables)} items):")
        print("-" * 80)
        for i, item in enumerate(tables[:1]):  # Show first 1
            print(f"[{i}] Type: {item['element_type']}")
            print(f"    HTML: {item['text_as_html'][:100]}..." if item['text_as_html'] and len(item['text_as_html']) > 100 else f"    HTML: {item['text_as_html']}")
        if len(tables) > 1:
            print(f"    ... and {len(tables) - 1} more items")
    
    # Load images
    images_file = example_folder / "images.json"
    if images_file.exists():
        with open(images_file, 'r', encoding='utf-8') as f:
            images = json.load(f)
        print(f"\n🖼️  Images ({len(images)} items):")
        print("-" * 80)
        for i, item in enumerate(images[:1]):  # Show first 1
            print(f"[{i}] Type: {item['element_type']}")
            print(f"    Page: {item['page_number']}")
            print(f"    Base64 length: {len(item['image_base64']) if item['image_base64'] else 0} chars")
        if len(images) > 1:
            print(f"    ... and {len(images) - 1} more items")
    else:
        print("\n🖼️  Images: None")
    
    print("\n" + "=" * 80)
else:
    print(f"Folder not found: {example_folder}")
    print("\nAvailable partitioned folders:")
    for folder in sorted(Path("Partitioned_Data").rglob("*")):
        if folder.is_dir() and list(folder.glob("*.json")):
            print(f"  - {folder.relative_to('Partitioned_Data')}")

📂 Loading data from: Partitioned_Data\Ngoai_ngu\Ngoai_ngu_2022_Quy_doi_CCTA

📄 Texts (23 items):
--------------------------------------------------------------------------------
[0] Type: Header
    Text: BỘ GIÁO DỤC VÀ ĐÀO TẠO TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ
[1] Type: Text
    Text: NỘI
    ... and 21 more items

📊 Tables (2 items):
--------------------------------------------------------------------------------
[0] Type: Table
    HTML: <table><tbody><tr><td>lơi nhận:</td><td></td><td>KT.</td><td>HIỆU</td><td>TRƯỞNG</td></tr><tr><td>Nh...
    ... and 1 more items

🖼️  Images: None

